In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
#  FloodNet — VGG-16 Semi-Supervised Training
#  Paper : "Flood or Non-Flooded" (Jackson et al., Water 2023, 15, 875)
#  Architecture: 13 conv + 5 maxpool + 3 dense | 138M params | 224×224
#  λ swept 0.0 → 1.0 (step 0.1) — best expected at λ=0.2
#  50 epochs per λ | Adam lr=0.0001 | batch=16 | 80/20 labeled split
# ╚══════════════════════════════════════════════════════════════════════╝

import os, copy, random, warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms, models
from torchvision.models import VGG16_Weights
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
)
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

# ── 1. PATHS ───────────────────────────────────────────────────────────
BASE = Path(
    "/kaggle/input/datasets/aletbm/"
    "aerial-imagery-dataset-floodnet-challenge/"
    "FloodNet Challenge - Track 1"
)
FLOODED_DIR    = BASE / "Train/Labeled/Flooded/image"
NONFLOODED_DIR = BASE / "Train/Labeled/Non-Flooded/image"
UNLABELED_DIR  = BASE / "Train/Unlabeled/image"
EXTS = {".jpg",".jpeg",".png",".JPG",".JPEG",".PNG"}

print("=== Path Check ===")
for name, p in [("Flooded", FLOODED_DIR),
                ("Non-Flooded", NONFLOODED_DIR),
                ("Unlabeled", UNLABELED_DIR)]:
    print(f"  {'OK' if Path(p).exists() else 'MISSING':7} {name:<14} → {p}")

# ── 2. Collect & split labeled data 80/20 ─────────────────────────────
def collect(folder, label):
    return [(str(p), label) for p in Path(folder).iterdir() if p.suffix in EXTS]

flooded_s    = collect(FLOODED_DIR,    1)   # 51
nonflooded_s = collect(NONFLOODED_DIR, 0)   # 347

random.shuffle(flooded_s)
random.shuffle(nonflooded_s)

def split80(lst):
    cut = int(0.8 * len(lst))
    return lst[:cut], lst[cut:]

f_train,  f_val  = split80(flooded_s)
nf_train, nf_val = split80(nonflooded_s)

TRAIN_SAMPLES   = f_train + nf_train
VAL_SAMPLES     = f_val   + nf_val
UNLABELED_PATHS = [str(p) for p in Path(UNLABELED_DIR).iterdir()
                   if p.suffix in EXTS]

n_f_tr  = len(f_train)
n_nf_tr = len(nf_train)
n_tr    = len(TRAIN_SAMPLES)

print(f"\n=== Dataset Split ===")
print(f"  Train     : {n_tr}  ({n_f_tr} flooded | {n_nf_tr} non-flooded)")
print(f"  Val       : {len(VAL_SAMPLES)}  ({len(f_val)} flooded | {len(nf_val)} non-flooded)")
print(f"  Unlabeled : {len(UNLABELED_PATHS)}")

# ── 3. Hyper-parameters ────────────────────────────────────────────────
IMG_SIZE  = 224          # VGG-16 native input: 224×224×3
BATCH     = 16
LR        = 1e-4
E         = 50
E_ia      = 20           # labeled-only phase ends (E_i^a)
E_fa      = 40           # alpha ramp ends          (E_f^a)
a_i, a_f  = 0.0, 1.0
DEVICE    = torch.device("cuda" if torch.cuda.is_available() else "cpu")

LAMBDA_VALUES = [round(x * 0.1, 1) for x in range(11)]   # 0.0 … 1.0

print(f"\nDevice  : {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU     : {torch.cuda.get_device_name(0)}")
    print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
print(f"λ sweep : {LAMBDA_VALUES}")
print(f"Total   : {len(LAMBDA_VALUES)} runs × {E} epochs = {len(LAMBDA_VALUES)*E} epochs")

# ── 4. Alpha schedule (Algorithm 1, lines 2-7) ────────────────────────
def get_alpha(ep):
    if ep < E_ia: return a_i
    if ep < E_fa: return ((a_f-a_i)/(E_fa-E_ia))*(ep-E_ia)+a_i
    return a_f

# ── 5. Transforms — 224×224 as per VGG-16 spec ────────────────────────
tfm = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),   # 224×224×3
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],
                         [0.229,0.224,0.225]),
])

# ── 6. Dataset classes ────────────────────────────────────────────────
class LabeledDS(Dataset):
    def __init__(self, samples, tfm=None):
        self.samples, self.tfm = samples, tfm
    def __len__(self): return len(self.samples)
    def __getitem__(self, i):
        path, lbl = self.samples[i]
        img = Image.open(path).convert("RGB")
        if self.tfm: img = self.tfm(img)
        return img, torch.tensor(lbl, dtype=torch.float32)

class UnlabeledDS(Dataset):
    def __init__(self, paths, tfm=None):
        self.paths, self.tfm = paths, tfm
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        img = Image.open(self.paths[i]).convert("RGB")
        if self.tfm: img = self.tfm(img)
        return img, self.paths[i]

# ── 7. Dataloaders (shared across all λ runs) ─────────────────────────
cw      = [n_tr/(2*n_nf_tr), n_tr/(2*n_f_tr)]
sw      = [cw[int(s[1])] for s in TRAIN_SAMPLES]
sampler = WeightedRandomSampler(sw, len(sw), replacement=True)

train_ds  = LabeledDS(TRAIN_SAMPLES,    tfm)
val_ds    = LabeledDS(VAL_SAMPLES,      tfm)
unl_ds    = UnlabeledDS(UNLABELED_PATHS, tfm)

train_ldr = DataLoader(train_ds, batch_size=BATCH, sampler=sampler,
                       num_workers=2, pin_memory=True)
val_ldr   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False,
                       num_workers=2, pin_memory=True)
unl_ldr   = DataLoader(unl_ds,   batch_size=BATCH, shuffle=False,
                       num_workers=2, pin_memory=True)

# ── 8. Build VGG-16 ───────────────────────────────────────────────────
def build_vgg16():
    """
    VGG-16 architecture:
      Features : 13 conv layers (3×3 kernel, stride 1, same padding)
                 Conv1: 64 filters  (×2)
                 Conv2: 128 filters (×2)
                 Conv3: 256 filters (×3)
                 Conv4: 512 filters (×3)
                 Conv5: 512 filters (×3)
                 5 MaxPool layers (2×2)
      Classifier: 3 FC layers
                  4096 → 4096 → 1000 (original)
                  Replace last layer → 1 (binary)
    Pretrained on ImageNet, fine-tuned for flood classification.
    """
    m = models.vgg16(weights=VGG16_Weights.IMAGENET1K_V1)

    # Replace only the final FC layer: 4096 → 1 (binary sigmoid)
    # Keep the first two FC layers (4096→4096) pretrained
    in_features = m.classifier[-1].in_features   # 4096
    m.classifier[-1] = nn.Linear(in_features, 1)

    return m

# Verify architecture once
_tmp = build_vgg16()
total_params = sum(p.numel() for p in _tmp.parameters())
trainable    = sum(p.numel() for p in _tmp.parameters() if p.requires_grad)
print(f"\n=== VGG-16 Architecture ===")
print(f"  Total params     : {total_params/1e6:.1f}M")
print(f"  Trainable params : {trainable/1e6:.1f}M")
print(f"  Input shape      : 3 × {IMG_SIZE} × {IMG_SIZE}")
print(f"  Output           : 1 node (BCEWithLogitsLoss)")
del _tmp

# ── 9. Evaluation ─────────────────────────────────────────────────────
def evaluate(model, loader):
    model.eval()
    probs_all, preds_all, lbls_all = [], [], []
    with torch.no_grad():
        for imgs, lbls in loader:
            logits = model(imgs.to(DEVICE)).squeeze(1)
            probs  = torch.sigmoid(logits).cpu().numpy()
            preds  = (probs >= 0.5).astype(int)
            probs_all.extend(probs)
            preds_all.extend(preds)
            lbls_all.extend(lbls.numpy().astype(int))
    acc  = accuracy_score(lbls_all, preds_all)
    f1   = f1_score(lbls_all,       preds_all, zero_division=0)
    prec = precision_score(lbls_all, preds_all, zero_division=0)
    rec  = recall_score(lbls_all,   preds_all, zero_division=0)
    try:    roc = roc_auc_score(lbls_all, probs_all)
    except: roc = float("nan")
    return acc, f1, prec, rec, roc

# ── 10. One full training run for a given λ ───────────────────────────
def train_one_lambda(lam):
    model     = build_vgg16().to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=LR)
    criterion = nn.BCEWithLogitsLoss()

    history    = []
    best_f1    = -1
    best_epoch = -1
    best_row   = {}
    best_state = None

    print(f"\n{'═'*88}")
    print(f"  VGG-16  |  λ = {lam:.1f}  |  {E} epochs")
    print(f"{'═'*88}")
    print(f"{'Ep':>3} │ {'λ':>4} │ {'α':>5} │ {'Loss':>8} │ "
          f"{'Acc':>6} │ {'F1':>6} │ {'Prec':>6} │ {'Rec':>6} │ {'AUC':>6}")
    print(f"{'─'*88}")

    for ep in range(E):
        model.train()
        alpha = get_alpha(ep)
        total_loss, n_batches = 0.0, 0

        # Phase A — labeled pass
        for imgs, lbls in train_ldr:
            imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(imgs).squeeze(1), lbls)
            loss.backward(); optimizer.step()
            total_loss += loss.item(); n_batches += 1

        # Phase B — score unlabeled → pseudo-labels (Algorithm 1 lines 8-15)
        #   p ≤ 0.5 − λ  →  non-flooded (0)  confident
        #   p ≥ 0.5 + λ  →  flooded     (1)  confident
        #   else          →  ignored          uncertain margin
        if alpha > 0:
            model.eval()
            p_imgs, p_lbls = [], []
            with torch.no_grad():
                for imgs, _ in unl_ldr:
                    imgs  = imgs.to(DEVICE)
                    probs = torch.sigmoid(
                                model(imgs).squeeze(1)
                            ).cpu().numpy()
                    for i, pv in enumerate(probs):
                        if pv <= 0.5 - lam:
                            p_imgs.append(imgs[i].cpu())
                            p_lbls.append(torch.tensor(0.0))
                        elif pv >= 0.5 + lam:
                            p_imgs.append(imgs[i].cpu())
                            p_lbls.append(torch.tensor(1.0))

            # Phase C — fine-tune on pseudo-labeled samples
            if p_imgs:
                model.train()
                for s in range(0, len(p_imgs), BATCH):
                    pi = torch.stack(p_imgs[s:s+BATCH]).to(DEVICE)
                    pl = torch.stack(p_lbls[s:s+BATCH]).to(DEVICE)
                    optimizer.zero_grad()
                    loss_u = alpha * criterion(model(pi).squeeze(1), pl)
                    loss_u.backward(); optimizer.step()
                    total_loss += loss_u.item(); n_batches += 1

        avg_loss = total_loss / max(n_batches, 1)
        acc, f1, prec, rec, roc = evaluate(model, val_ldr)

        row = dict(lambda_=lam, epoch=ep+1, alpha=round(alpha,4),
                   loss=round(avg_loss,4), accuracy=round(acc,4),
                   f1=round(f1,4), precision=round(prec,4),
                   recall=round(rec,4), roc_auc=round(roc,4))
        history.append(row)

        print(f"{ep+1:3d} │ {lam:4.1f} │ {alpha:5.3f} │ {avg_loss:8.4f} │ "
              f"{acc:6.4f} │ {f1:6.4f} │ {prec:6.4f} │ {rec:6.4f} │ {roc:6.4f}")

        if f1 > best_f1:
            best_f1=f1; best_epoch=ep+1
            best_row=row.copy()
            best_state=copy.deepcopy(model.state_dict())

    print(f"{'─'*88}")
    print(f"  ★ λ={lam:.1f}  Best epoch {best_epoch}  |  "
          f"Acc={best_row['accuracy']:.4f}  F1={best_row['f1']:.4f}  "
          f"Prec={best_row['precision']:.4f}  Rec={best_row['recall']:.4f}  "
          f"AUC={best_row['roc_auc']:.4f}")

    # Free GPU memory between λ runs
    del model
    torch.cuda.empty_cache()

    return history, best_row, best_state

# ── 11. λ SWEEP ───────────────────────────────────────────────────────
print("\n\n" + "█"*88)
print("  STARTING λ SWEEP  —  VGG-16  |  FloodNet Semi-Supervised")
print("█"*88)

all_history  = []
summary_rows = []
best_states  = {}

for lam in LAMBDA_VALUES:
    hist, best_row, best_state = train_one_lambda(lam)
    all_history.extend(hist)
    summary_rows.append(best_row)
    best_states[lam] = best_state

# ── 12. Summary table ─────────────────────────────────────────────────
summary_df = pd.DataFrame(summary_rows).sort_values("lambda_").reset_index(drop=True)
all_df     = pd.DataFrame(all_history)

print("\n\n" + "═"*88)
print("  FINAL SUMMARY — VGG-16 | Best results per λ value")
print("═"*88)
print(f"{'λ':>5} │ {'Epoch':>5} │ {'Acc':>6} │ {'F1':>6} │ "
      f"{'Prec':>6} │ {'Rec':>6} │ {'AUC':>6}")
print("─"*88)
best_f1_overall = summary_df["f1"].max()
for _, r in summary_df.iterrows():
    marker = "  ◄ BEST" if r["f1"] == best_f1_overall else ""
    print(f"{r['lambda_']:>5.1f} │ {int(r['epoch']):>5} │ "
          f"{r['accuracy']:>6.4f} │ {r['f1']:>6.4f} │ "
          f"{r['precision']:>6.4f} │ {r['recall']:>6.4f} │ "
          f"{r['roc_auc']:>6.4f}{marker}")
print("═"*88)

best_lambda  = summary_df.loc[summary_df["f1"].idxmax(), "lambda_"]
best_overall = summary_df.loc[summary_df["f1"].idxmax()]
print(f"\n  ★ Best λ       : {best_lambda}")
print(f"  ★ Best Epoch   : {int(best_overall['epoch'])}")
print(f"  ★ Accuracy     : {best_overall['accuracy']:.4f}")
print(f"  ★ F1 Score     : {best_overall['f1']:.4f}")
print(f"  ★ Precision    : {best_overall['precision']:.4f}")
print(f"  ★ Recall       : {best_overall['recall']:.4f}")
print(f"  ★ ROC-AUC      : {best_overall['roc_auc']:.4f}")

# ── 13. Save ──────────────────────────────────────────────────────────
all_df.to_csv("/kaggle/working/vgg16_all_history.csv",        index=False)
summary_df.to_csv("/kaggle/working/vgg16_lambda_summary.csv", index=False)
torch.save(best_states[best_lambda], "/kaggle/working/vgg16_best.pth")
print(f"\nSaved → vgg16_all_history.csv")
print(f"Saved → vgg16_lambda_summary.csv")
print(f"Saved → vgg16_best.pth  (λ={best_lambda})")

# ── 14. Plot 1 — metrics vs λ ─────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
fig.suptitle("VGG-16  |  FloodNet  |  Best metrics vs λ",
             fontsize=13, fontweight="bold")
metric_pairs = [
    ("accuracy","Accuracy","tab:blue"),
    ("f1","F1 Score","tab:green"),
    ("precision","Precision","tab:orange"),
    ("recall","Recall","tab:purple"),
    ("roc_auc","ROC-AUC","tab:brown"),
    ("epoch","Best Epoch Found","tab:red"),
]
for ax, (col, title, color) in zip(axes.flat, metric_pairs):
    ax.plot(summary_df["lambda_"], summary_df[col],
            color=color, linewidth=2, marker="o", markersize=7)
    if col != "epoch":
        best_lam_for_col = summary_df.loc[summary_df[col].idxmax(), "lambda_"]
        ax.axvline(best_lam_for_col, color="black", linestyle="--",
                   linewidth=1, alpha=0.6, label=f"Peak λ={best_lam_for_col}")
        ax.legend(fontsize=8)
    ax.set_title(title, fontsize=11)
    ax.set_xlabel("λ (uncertainty offset)")
    ax.set_ylabel(title)
    ax.set_xticks(LAMBDA_VALUES)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("/kaggle/working/vgg16_metrics_vs_lambda.png",
            dpi=150, bbox_inches="tight")
plt.show()
print("Saved → vgg16_metrics_vs_lambda.png")

# ── 15. Plot 2 — training curves for best λ + all-λ F1 overlay ────────
best_hist = all_df[all_df["lambda_"] == best_lambda].reset_index(drop=True)
best_ep   = int(best_overall["epoch"])

fig2, axes2 = plt.subplots(2, 3, figsize=(16, 8))
fig2.suptitle(
    f"VGG-16  |  FloodNet  |  λ={best_lambda} (best)  |  Best epoch={best_ep}",
    fontsize=13, fontweight="bold"
)
for ax, (col, title, color) in zip(axes2.flat, metric_pairs[:-1]):
    ax.plot(best_hist["epoch"], best_hist[col], color=color, linewidth=2)
    ax.axvline(best_ep, color="black", linestyle="--",
               linewidth=1.2, label=f"Best ep {best_ep}")
    bv = best_hist.loc[best_hist["epoch"]==best_ep, col].values[0]
    ax.scatter([best_ep],[bv], color="black", zorder=5, s=60)
    ax.axvline(E_ia, color="gray", linestyle=":", linewidth=1,
               alpha=0.5, label=f"Phase boundary")
    ax.axvline(E_fa, color="gray", linestyle=":", linewidth=1, alpha=0.5)
    ax.set_title(title, fontsize=11)
    ax.set_xlabel("Epoch")
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)

# Last panel — F1 for all λ overlaid
ax_last = axes2.flat[-1]
colors_sweep = plt.cm.viridis(np.linspace(0, 1, len(LAMBDA_VALUES)))
for lam, col in zip(LAMBDA_VALUES, colors_sweep):
    sub = all_df[all_df["lambda_"]==lam]
    lw  = 2.5 if lam == best_lambda else 1.0
    ax_last.plot(sub["epoch"], sub["f1"],
                 color=col, linewidth=lw,
                 label=f"λ={lam}" + (" ★" if lam==best_lambda else ""))
ax_last.set_title("F1 Score — all λ values", fontsize=11)
ax_last.set_xlabel("Epoch")
ax_last.set_ylabel("F1 Score")
ax_last.legend(fontsize=6, ncol=2)
ax_last.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("/kaggle/working/vgg16_training_curves.png",
            dpi=150, bbox_inches="tight")
plt.show()
print("Saved → vgg16_training_curves.png")

print("\n✓ VGG-16 λ sweep complete.")